02_text_train.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1zOVs8kDMUh1PLKtHieAkL9zh5tcgWtC9

# DeepTrace — Module 2: Text Fine-Tuning (FIXED)
## DeBERTa-v3-xsmall | T4 GPU | ~60-90 min

### Why the previous run gave F1=0.000:
- LR was 8e-6 — far too small for warming up from scratch
- No class weighting — model predicted majority class only
- `fp16=False` but no compensating fix — gradients were too small

### Fixes applied:
1. **Two-phase training**: warm up head at LR=2e-4, then full fine-tune at LR=3e-5
2. **Weighted CrossEntropy** to handle class imbalance
3. **fp16=True** for T4 GPU speed + gradient accumulation
4. **Longer training**: 5 epochs total with early stopping
5. **Verified forward pass** before full training loop

### Runtime: T4 GPU (Runtime → Change runtime type → T4)

In [ ]:
# CELL 1 — Install
!pip install -q transformers datasets accelerate scikit-learn sentencepiece protobuf

In [ ]:
# CELL 2 — GPU check
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
DEVICE = 'cuda'

In [ ]:
# CELL 3 — Upload processed zip
from google.colab import files
import zipfile, os

print('Upload text_processed.zip from notebook 01')
uploaded = files.upload()
os.makedirs('/content/data/processed', exist_ok=True)
with zipfile.ZipFile('text_processed.zip','r') as z:
    z.extractall('/content/data/processed/')
print('Files:')
import glob as _g, os as _os
for _f in sorted(_g.glob('/content/data/processed/*')):
    print(f'  {_os.path.basename(_f)}  ({_os.path.getsize(_f)//1024} KB)')

In [ ]:
# CELL 4 — Config (T4-optimised)
MODEL_NAME   = 'microsoft/deberta-v3-xsmall'
MAX_LEN      = 128    # T4 safe at 128; use 256 if you have A100
BATCH_SIZE   = 16     # T4: 16 at 128 tokens
GRAD_ACCUM   = 2      # effective batch = 32

# Phase 1: train classification head only (frozen backbone)
LR_HEAD      = 2e-4
EPOCHS_HEAD  = 1

# Phase 2: unfreeze all, full fine-tune
LR_FULL      = 3e-5   # FIXED: 3e-5 is correct (8e-6 was causing F1=0)
EPOCHS_FULL  = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# Subset size (keeps T4 runtime manageable)
TRAIN_LIMIT  = 30000
VAL_LIMIT    = 4000
TEST_LIMIT   = 4000

print('Config OK')

In [ ]:
# CELL 5 — Load data + verify both classes present
import pandas as pd
from collections import Counter

def load_split(path, name, limit):
    df = pd.read_csv(path)
    df = df.dropna(subset=['text','label'])
    df['text']  = df['text'].astype(str).str.strip()
    df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
    df = df[df['label'].isin([0,1])]
    df = df[df['text'].str.len() > 2].drop_duplicates('text').reset_index(drop=True)

    # Stratified sample
    if len(df) > limit:
        parts = []
        for lv, g in df.groupby('label'):
            n = max(1, int(round(len(g)/len(df)*limit)))
            parts.append(g.sample(n=min(n,len(g)), random_state=42))
        df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f'{name}: {len(df)}  labels={Counter(df.label)}')
    # CRITICAL: assert both classes exist
    assert 0 in df.label.values and 1 in df.label.values, \
        f'Missing class in {name}! Check 01_text_preprocess output.'
    return df

train_df = load_split('/content/data/processed/text_train.csv', 'train', TRAIN_LIMIT)
val_df   = load_split('/content/data/processed/text_val.csv',   'val',   VAL_LIMIT)
test_df  = load_split('/content/data/processed/text_test.csv',  'test',  TEST_LIMIT)

# Compute class weight for loss function
n0 = (train_df.label==0).sum()
n1 = (train_df.label==1).sum()
pos_weight = n0 / max(n1, 1)
print(f'\npos_weight (legit/phish ratio): {pos_weight:.3f}')
print('This will be used in weighted loss to prevent class bias.')

In [ ]:
# CELL 6 — Tokenize
from datasets import Dataset
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def make_hf_dataset(df):
    return Dataset.from_dict({
        'text':  df['text'].astype(str).tolist(),
        'label': df['label'].astype(int).tolist()
    })

def tokenize_fn(batch):
    enc = tokenizer(
        [str(t) for t in batch['text']],
        truncation=True, padding='max_length',
        max_length=MAX_LEN
    )
    # ensure labels are clean int
    enc['labels'] = [int(x) if int(x) in [0,1] else 0 for x in batch['label']]
    return enc

COLS = ['input_ids','attention_mask','labels']

train_tok = make_hf_dataset(train_df).map(tokenize_fn, batched=True, remove_columns=['text','label'])
val_tok   = make_hf_dataset(val_df).map(tokenize_fn,   batched=True, remove_columns=['text','label'])
test_tok  = make_hf_dataset(test_df).map(tokenize_fn,  batched=True, remove_columns=['text','label'])

train_tok.set_format(type='torch', columns=COLS)
val_tok.set_format(  type='torch', columns=COLS)
test_tok.set_format( type='torch', columns=COLS)

# Verify
s = train_tok[0]
assert s['input_ids'].dtype == torch.long
assert s['labels'].item() in [0,1]
label_counts = Counter([train_tok[i]['labels'].item() for i in range(min(200, len(train_tok)))])
assert 0 in label_counts and 1 in label_counts, f'Class missing in tokenized data: {label_counts}'
print(f'✓ Tokenization OK. Sample label counts (first 200): {label_counts}')

In [ ]:
# CELL 7 — Load model
import torch
from transformers import AutoModelForSequenceClassification

print(f'Loading {MODEL_NAME}...')
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        problem_type='single_label_classification',
        attn_implementation='eager'
    )
except TypeError:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        problem_type='single_label_classification'
    )

model.config.use_cache = False
model = model.to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,}')

In [ ]:
# CELL 8 — Forward-pass sanity check (catches NaN before wasting GPU time)
from torch.utils.data import DataLoader

loader = DataLoader(train_tok, batch_size=4, shuffle=True)
batch  = next(iter(loader))
batch  = {k: v.to(DEVICE) for k,v in batch.items()}

model.eval()
with torch.no_grad():
    out = model(**batch)

print(f'Loss        : {out.loss.item():.4f}')
print(f'Finite loss : {torch.isfinite(out.loss).item()}')
print(f'Finite logits: {torch.isfinite(out.logits).all().item()}')

assert torch.isfinite(out.loss),         'Forward pass loss is NaN — check data'
assert torch.isfinite(out.logits).all(), 'Forward pass logits have NaN — check model'
# Loss check: expect ~0.693 at random init (informational only)
print(f'Initial loss: {out.loss.item():.4f} (expected ~0.693 at random init)')

print('✓ Sanity check passed')
model.train()

In [ ]:
# CELL 9 — Standard Metrics (Data is balanced, no custom loss needed)
import numpy as np
import torch
from transformers import Trainer, TrainingArguments, DefaultDataCollator
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.clip(np.nan_to_num(np.asarray(logits, dtype=np.float32), nan=0.0), -20, 20)
    labels = np.asarray(labels, dtype=np.int64)
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:,1]
    probs  = np.nan_to_num(probs, nan=0.5)
    preds  = np.argmax(logits, axis=-1)

    out = {
        'acc': float(accuracy_score(labels, preds)),
        'f1':  float(f1_score(labels, preds, zero_division=0)),
    }
    if len(np.unique(labels)) == 2:
        out['auc'] = float(roc_auc_score(labels, probs))
    else:
        out['auc'] = 0.5
    return out

print('Metrics defined.')

In [ ]:
# CELL 10 — PHASE 1: Train head only (frozen backbone)
from transformers import Trainer # Ensure standard Trainer is imported

# Freeze all backbone parameters
for name, param in model.named_parameters():
    if 'classifier' not in name and 'pooler' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Phase 1: training {trainable:,} / {total_params:,} params (head only)')

args_head = TrainingArguments(
    output_dir            = '/content/ckpt_head',
    eval_strategy         = 'epoch',
    save_strategy         = 'no',
    logging_strategy      = 'steps',
    logging_steps         = 50,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs      = EPOCHS_HEAD,
    learning_rate         = LR_HEAD,
    weight_decay          = WEIGHT_DECAY,
    max_grad_norm         = 1.0,
    fp16                  = False,    # Stable FP32
    bf16                  = False,
    report_to             = 'none',
    dataloader_num_workers= 2,
    remove_unused_columns = False,
    warmup_ratio          = WARMUP_RATIO,
)

trainer_head = Trainer(  # <--- KEY FIX: Using Standard Trainer
    model        = model,
    args         = args_head,
    train_dataset= train_tok,
    eval_dataset = val_tok,
    data_collator= DefaultDataCollator(),
    compute_metrics = compute_metrics,
)

print('Phase 1 training...')
trainer_head.train()
m = trainer_head.evaluate()
print(f'Phase 1 done — Acc={m["eval_acc"]:.4f}  F1={m["eval_f1"]:.4f}  AUC={m.get("eval_auc",0):.4f}')
assert m['eval_f1'] > 0.1, f'F1 still ~0 after Phase 1 ({m["eval_f1"]:.4f}). Check data balance.'

In [ ]:
# CELL 11 — Full fine-tune (Straight-through, safely configured)
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Training ALL {trainable:,} params')

args_full = TrainingArguments(
    output_dir            = '/content/ckpt_full',
    eval_strategy         = 'epoch',
    save_strategy         = 'epoch',
    save_total_limit      = 1,
    load_best_model_at_end= True,
    metric_for_best_model = 'eval_f1',
    greater_is_better     = True,
    logging_strategy      = 'steps',
    logging_steps         = 100,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs      = EPOCHS_FULL,
    learning_rate         = 1e-6,    # STRICT CAP: Prevents gradient explosion
    weight_decay          = WEIGHT_DECAY,
    max_grad_norm         = 1.0,
    fp16                  = False,   # MUST REMAIN FALSE
    bf16                  = False,
    adam_epsilon          = 1e-6,    # CRITICAL FIX: Stops DeBERTa from generating NaNs
    report_to             = 'none',
    dataloader_num_workers= 2,
    remove_unused_columns = False,
    warmup_ratio          = WARMUP_RATIO,
)

from transformers import EarlyStoppingCallback, Trainer

trainer_full = Trainer(
    model        = model,
    args         = args_full,
    train_dataset= train_tok,
    eval_dataset = val_tok,
    data_collator= DefaultDataCollator(),
    compute_metrics = compute_metrics,
    callbacks    = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Training...')
trainer_full.train()
print('Training complete.')

In [ ]:
# CELL 12 — Final evaluation (val + test) with detailed metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# -----------------------------
# 1) Evaluate using Trainer API
# -----------------------------
val_m  = trainer_full.evaluate(eval_dataset=val_tok)
test_m = trainer_full.evaluate(eval_dataset=test_tok)

print('=' * 72)
print('FINAL MODEL EVALUATION')
print('=' * 72)

print('\n[VALIDATION METRICS]')
print(f'  Accuracy : {val_m.get("eval_acc", 0):.4f}')
print(f'  F1-Score : {val_m.get("eval_f1", 0):.4f}')
print(f'  AUC      : {val_m.get("eval_auc", 0):.4f}')

print('\n[TEST METRICS]')
print(f'  Accuracy : {test_m.get("eval_acc", 0):.4f}')
print(f'  F1-Score : {test_m.get("eval_f1", 0):.4f}')
print(f'  AUC      : {test_m.get("eval_auc", 0):.4f}')

# -----------------------------
# 2) Get raw predictions
# -----------------------------
pred_output = trainer_full.predict(test_tok)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=1)

# Stable softmax to avoid overflow
logits_shifted = logits - np.max(logits, axis=1, keepdims=True)
exp_logits = np.exp(logits_shifted)
y_prob = exp_logits / exp_logits.sum(axis=1, keepdims=True)

# -----------------------------
# 3) Extra metrics
# -----------------------------
test_acc  = accuracy_score(y_true, y_pred)
test_f1   = f1_score(y_true, y_pred, zero_division=0)
test_prec = precision_score(y_true, y_pred, zero_division=0)
test_rec  = recall_score(y_true, y_pred, zero_division=0)

if len(np.unique(y_true)) == 2:
    test_auc = roc_auc_score(y_true, y_prob[:, 1])
else:
    test_auc = 0.0

print('\n' + '=' * 72)
print('DETAILED TEST PERFORMANCE')
print('=' * 72)
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  Precision : {test_prec:.4f}')
print(f'  Recall    : {test_rec:.4f}')
print(f'  F1-Score  : {test_f1:.4f}')
print(f'  AUC       : {test_auc:.4f}')

# -----------------------------
# 4) Confusion Matrix
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

print('\nCONFUSION MATRIX')
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Phishing'])
disp.plot(cmap='Blues', values_format='d', ax=ax)
plt.title('Confusion Matrix - Test Set')
plt.grid(False)
plt.show()

# -----------------------------
# 5) Classification report
# -----------------------------
print('\nCLASSIFICATION REPORT')
print(classification_report(y_true, y_pred, digits=4))

# -----------------------------
# 6) Generalization gap
# -----------------------------
val_acc = val_m.get("eval_acc", 0)
val_f1  = val_m.get("eval_f1", 0)

print('=' * 72)
print('GENERALIZATION CHECK')
print('=' * 72)
print(f'  Accuracy Gap (Val - Test) : {val_acc - test_acc:.4f}')
print(f'  F1 Gap       (Val - Test) : {val_f1 - test_f1:.4f}')

if abs(val_f1 - test_f1) < 0.03:
    print('  Status : Excellent generalization. No major overfitting visible.')
elif abs(val_f1 - test_f1) < 0.07:
    print('  Status : Slight generalization gap, but still acceptable.')
else:
    print('  Status : Noticeable gap. Recheck split quality and preprocessing.')

# -----------------------------
# 7) Save important variables
# -----------------------------
test_pred_labels = y_pred
test_pred_probs  = y_prob[:, 1]

print('\n✅ Evaluation complete.')

In [ ]:
# CELL 13 — Sample predictions with confidence analysis
import pandas as pd
import numpy as np

label_map = {0: 'Legitimate', 1: 'Phishing'}

# Build result dataframe from original test set
results_df = test_df.copy().reset_index(drop=True)
results_df['true_label_id'] = y_true
results_df['pred_label_id'] = test_pred_labels
results_df['confidence_class1'] = test_pred_probs
results_df['predicted_label'] = results_df['pred_label_id'].map(label_map)
results_df['actual_label'] = results_df['true_label_id'].map(label_map)
results_df['correct'] = results_df['true_label_id'] == results_df['pred_label_id']

# Confidence score of predicted class
results_df['prediction_confidence'] = np.where(
    results_df['pred_label_id'] == 1,
    results_df['confidence_class1'],
    1 - results_df['confidence_class1']
)

# Short preview text for neat display
results_df['text_preview'] = results_df['text'].astype(str).str.slice(0, 160)

print('=' * 90)
print('SAMPLE PREDICTIONS')
print('=' * 90)

# 1) Correct high-confidence predictions
print('\nTOP 5 HIGH-CONFIDENCE CORRECT PREDICTIONS')
top_correct = results_df[results_df['correct']].sort_values(
    by='prediction_confidence', ascending=False
).head(5)

display(top_correct[[
    'text_preview',
    'actual_label',
    'predicted_label',
    'prediction_confidence'
]].style.format({'prediction_confidence': '{:.4f}'}))

# 2) Wrong predictions for error analysis
print('\nTOP MISCLASSIFIED SAMPLES (IF ANY)')
wrong_df = results_df[~results_df['correct']].sort_values(
    by='prediction_confidence', ascending=False
).head(5)

if len(wrong_df) > 0:
    display(wrong_df[[
        'text_preview',
        'actual_label',
        'predicted_label',
        'prediction_confidence'
    ]].style.format({'prediction_confidence': '{:.4f}'}))
else:
    print('No misclassified samples found in the displayed subset.')

# 3) Overall prediction summary
print('\nPREDICTION SUMMARY')
print(f"  Total Test Samples : {len(results_df)}")
print(f"  Correct Predictions: {results_df['correct'].sum()}")
print(f"  Wrong Predictions  : {(~results_df['correct']).sum()}")
print(f"  Average Confidence : {results_df['prediction_confidence'].mean():.4f}")

print('\nCLASS-WISE PREDICTION COUNTS')
print(results_df['predicted_label'].value_counts())

print('\n✅ Sample prediction analysis ready.')

In [ ]:
# CELL 14 — Save model + download
import os, json, shutil
from google.colab import files

SAVE_DIR = '/content/deberta_phish'
os.makedirs(SAVE_DIR, exist_ok=True)

trainer_full.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

cfg = {
    'model_name':  MODEL_NAME,
    'max_len':     MAX_LEN,
    'num_labels':  2,
    'batch_size':  BATCH_SIZE,
    'lr_full':     LR_FULL,
    'pos_weight':  float(pos_weight),
    'val_acc':     float(val_m.get('eval_acc',0)),
    'val_f1':      float(val_m.get('eval_f1',0)),
    'val_auc':     float(val_m.get('eval_auc',0)),
    'test_acc':    float(test_m.get('eval_acc',0)),
    'test_f1':     float(test_m.get('eval_f1',0)),
    'test_auc':    float(test_m.get('eval_auc',0)),
}
with open(f'{SAVE_DIR}/deeptrace_config.json','w') as f:
    json.dump(cfg, f, indent=2)

print('Saved files:')
import glob as _g2, os as _o2
for _fp in sorted(_g2.glob(SAVE_DIR+'/*')):
    print(f'  {_o2.path.basename(_fp)}  ({_o2.path.getsize(_fp)//1024} KB)')

shutil.make_archive('/content/text_model_pkg','zip','/content','deberta_phish')
files.download('/content/text_model_pkg.zip')
print('\n✅ Extract into: DeepTrace/training/module2_text/models/')